In [63]:
df_inventory = spark.read.parquet("abfss://22aa6652-1cad-462b-96e1-9ad0e44f9e9a@onelake.dfs.fabric.microsoft.com/6e23c5d3-4cf2-4a19-baa8-e8b89b7cdfad/Files/Bronze/inventory_data.parquet")


StatementMeta(, 2e424b1e-5c63-404a-b421-0f94eac6cf48, 42, Finished, Available, Finished, False)

In [64]:
display(df_inventory)

StatementMeta(, 2e424b1e-5c63-404a-b421-0f94eac6cf48, 43, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 0ef78b10-8de5-4898-8246-b47ded02db5c)

In [65]:
df_order = spark.read.parquet("abfss://22aa6652-1cad-462b-96e1-9ad0e44f9e9a@onelake.dfs.fabric.microsoft.com/6e23c5d3-4cf2-4a19-baa8-e8b89b7cdfad/Files/Bronze/orders_data.parquet")

StatementMeta(, 2e424b1e-5c63-404a-b421-0f94eac6cf48, 44, Finished, Available, Finished, False)

In [66]:
display(df_order)

StatementMeta(, 2e424b1e-5c63-404a-b421-0f94eac6cf48, 45, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, bd7d8626-59ce-4d43-b3f0-815e7b508431)

In [67]:
df_returns = spark.read.parquet("abfss://22aa6652-1cad-462b-96e1-9ad0e44f9e9a@onelake.dfs.fabric.microsoft.com/6e23c5d3-4cf2-4a19-baa8-e8b89b7cdfad/Files/Bronze/returns_data.xlsx.parquet")

StatementMeta(, 2e424b1e-5c63-404a-b421-0f94eac6cf48, 46, Finished, Available, Finished, False)

In [68]:
display(df_returns)

StatementMeta(, 2e424b1e-5c63-404a-b421-0f94eac6cf48, 47, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, d6bf6a26-82e5-425f-8320-fd4c0c7b5b5e)

In [73]:
from pyspark.sql.functions import *

# ============================================================
# ORDERS - CLEANING FOR YOUR ACTUAL TABLE
# ============================================================

# Start from the ORIGINAL raw Orders dataframe
# Your actual dataframe is df_order
df_orders = df_order


# ============================================================
# 1. Rename columns to clean standard names
# ============================================================

df_orders = (
    df_orders

    # cust_id → CustomerID
    .withColumnRenamed(
        "cust_id",
        "CustomerID"
    )

    # Product_Name → ProductName
    .withColumnRenamed(
        "Product_Name",
        "ProductName"
    )

    # Order_Amount$ → OrderAmount
    .withColumnRenamed(
        "Order_Amount$",
        "OrderAmount"
    )
)


# ============================================================
# 2. CLEAN ORDER DATE
# ============================================================
# Your data contains formats such as:
#
# 2023/06/12
# 12-07-2023
# 2023.07.15
# 15/07/2023
# 07-20-2023
#
# Convert all of them to Spark DateType.
# ============================================================

df_orders = df_orders.withColumn(
    "Order_Date",
    coalesce(

        # 2023/06/12
        to_date(
            col("Order_Date"),
            "yyyy/MM/dd"
        ),

        # 12-07-2023
        to_date(
            col("Order_Date"),
            "dd-MM-yyyy"
        ),

        # 2023.07.15
        to_date(
            col("Order_Date"),
            "yyyy.MM.dd"
        ),

        # 15/07/2023
        to_date(
            col("Order_Date"),
            "dd/MM/yyyy"
        ),

        # 07-20-2023
        to_date(
            col("Order_Date"),
            "MM-dd-yyyy"
        ),

        # 2023-07-15
        to_date(
            col("Order_Date"),
            "yyyy-MM-dd"
        )
    )
)


# ============================================================
# 3. STANDARDIZE CUSTOMER ID
# ============================================================

df_orders = df_orders.withColumn(
    "CustomerID",
    trim(
        upper(
            col("CustomerID")
        )
    )
)


# ============================================================
# 4. CLEAN PRODUCT NAME
# ============================================================

df_orders = df_orders.withColumn(
    "ProductName",
    initcap(
        trim(
            col("ProductName")
        )
    )
)


# ============================================================
# 5. CLEAN QUANTITY
# ============================================================
# Handles:
# 1
# 3
# one
# ONE
# two
# etc.
# ============================================================

df_orders = df_orders.withColumn(
    "Qty",
    when(
        col("Qty").cast("int").isNotNull(),
        col("Qty").cast("int")
    )
    .when(
        lower(trim(col("Qty"))) == "one",
        lit(1)
    )
    .when(
        lower(trim(col("Qty"))) == "two",
        lit(2)
    )
    .when(
        lower(trim(col("Qty"))) == "three",
        lit(3)
    )
    .when(
        lower(trim(col("Qty"))) == "four",
        lit(4)
    )
    .when(
        lower(trim(col("Qty"))) == "five",
        lit(5)
    )
    .otherwise(
        lit(None).cast("int")
    )
)


# ============================================================
# 6. CLEAN ORDER AMOUNT
# ============================================================
# Handles:
# $799.0
# $1200
# 499 USD
# ₹49
# ============================================================

df_orders = df_orders.withColumn(
    "OrderAmount",
    abs(
        regexp_extract(
            col("OrderAmount").cast("string"),
            r"(\d+(?:\.\d+)?)",
            1
        ).cast("double")
    )
)


# ============================================================
# 7. CLEAN EMAIL
# ============================================================

df_orders = df_orders.withColumn(
    "Email",
    when(
        col("Email").isNull(),
        lit(None)
    )
    .otherwise(
        lower(
            trim(
                col("Email")
            )
        )
    )
)


# ============================================================
# 8. CLEAN PAYMENT MODE
# ============================================================

df_orders = df_orders.withColumn(
    "Payment_Mode",
    when(
        col("Payment_Mode").isNull() |
        (trim(col("Payment_Mode")) == ""),
        lit("unknown")
    )
    .otherwise(
        lower(
            trim(
                col("Payment_Mode")
            )
        )
    )
)


# ============================================================
# 9. CLEAN PROMO CODE
# ============================================================

df_orders = df_orders.withColumn(
    "Promo_Code",
    when(
        col("Promo_Code").isNull() |
        (trim(col("Promo_Code")) == ""),
        lit("NO_PROMO")
    )
    .otherwise(
        upper(
            trim(
                col("Promo_Code")
            )
        )
    )
)


# ============================================================
# 10. CLEAN DELIVERY STATUS
# ============================================================

df_orders = df_orders.withColumn(
    "Delivery_Status",
    lower(
        trim(
            col("Delivery_Status")
        )
    )
)


# ============================================================
# 11. CLEAN SHIP ADDRESS
# ============================================================

df_orders = df_orders.withColumn(
    "Ship_Address",
    initcap(
        trim(
            regexp_replace(
                col("Ship_Address"),
                r"[^a-zA-Z0-9\s,#-]",
                " "
            )
        )
    )
)


# ============================================================
# 12. FEEDBACK SCORE
# ============================================================

df_orders = df_orders.withColumn(
    "Feedback_Score",
    col("Feedback_Score").cast("double")
)


# ============================================================
# 13. ADD YEAR AND MONTH
# ============================================================

df_orders = df_orders.withColumn(
    "Order_Year",
    year(
        col("Order_Date")
    )
)

df_orders = df_orders.withColumn(
    "Order_Month",
    month(
        col("Order_Date")
    )
)


# ============================================================
# 14. CREATE ORDER HASH
# ============================================================

df_orders = df_orders.withColumn(
    "Order_Hash",
    sha2(
        concat_ws(
            "|",
            *[
                coalesce(
                    col(c).cast("string"),
                    lit("")
                )
                for c in df_orders.columns
            ]
        ),
        256
    )
)


# ============================================================
# 15. REMOVE INVALID RECORDS
# ============================================================

df_orders = df_orders.dropna(
    subset=[
        "Order_ID",
        "CustomerID",
        "OrderAmount"
    ]
)


# ============================================================
# 16. REMOVE DUPLICATE ORDERS
# ============================================================

df_orders = df_orders.dropDuplicates(
    ["Order_ID"]
)


# ============================================================
# 17. DISPLAY CLEANED DATA
# ============================================================

print("========== CLEANED ORDERS ==========")

display(df_orders)


# ============================================================
# 18. CHECK SCHEMA
# ============================================================

print("========== ORDERS SCHEMA ==========")

df_orders.printSchema()


# ============================================================
# 19. CHECK DATE RESULTS
# ============================================================

print("========== ORDER DATES ==========")

display(
    df_orders.select(
        "Order_ID",
        "Order_Date"
    ).orderBy(
        "Order_Date"
    )
)


# ============================================================
# 20. SAVE SILVER ORDERS
# ============================================================

df_orders.write \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .format("delta") \
    .saveAsTable("silver_orders")

print("==========================================")
print("✅ silver_orders created successfully")
print("==========================================")

StatementMeta(, 2e424b1e-5c63-404a-b421-0f94eac6cf48, 52, Finished, Available, Finished, False)

========== CLEANED ORDERS ==========


SynapseWidget(Synapse.DataFrame, 1d3b8068-9533-4a0f-9e29-38d83fe2d7ed)

========== ORDERS SCHEMA ==========
root
 |-- Order_ID: string (nullable = true)
 |-- CustomerID: string (nullable = true)
 |-- ProductName: string (nullable = true)
 |-- Qty: integer (nullable = true)
 |-- Order_Date: date (nullable = true)
 |-- OrderAmount: double (nullable = true)
 |-- Delivery_Status: string (nullable = true)
 |-- Payment_Mode: string (nullable = true)
 |-- Ship_Address: string (nullable = true)
 |-- Email: string (nullable = true)
 |-- Promo_Code: string (nullable = true)
 |-- Feedback_Score: double (nullable = true)
 |-- Order_Year: integer (nullable = true)
 |-- Order_Month: integer (nullable = true)
 |-- Order_Hash: string (nullable = true)

========== ORDER DATES ==========


SynapseWidget(Synapse.DataFrame, 1d69f7bb-009e-4053-a7d2-8a93c4779f3e)

✅ silver_orders created successfully


In [74]:
df_orders_silver = spark.table("silver_orders")

display(df_orders_silver)

StatementMeta(, 2e424b1e-5c63-404a-b421-0f94eac6cf48, 53, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, aa968da5-313b-404d-b9a5-219355a8a291)

In [76]:
from pyspark.sql.functions import *

# Your actual raw DataFrame
df_inventory_cleaned = df_inventory

# Rename raw columns first
df_inventory_cleaned = (
    df_inventory_cleaned
    .withColumnRenamed("product_id", "ProductID")
    .withColumnRenamed("productName", "ProductName")
    .withColumnRenamed("stock", "RawStock")
    .withColumnRenamed("last_stocked", "LastStocked")
    .withColumnRenamed("warehouse", "RawWarehouse")
    .withColumnRenamed("cost_price", "CostPrice")
    .withColumnRenamed("available", "RawAvailable")
)

# Product name
df_inventory_cleaned = df_inventory_cleaned.withColumn(
    "ProductName",
    initcap(trim(col("ProductName")))
)

# Stock
df_inventory_cleaned = df_inventory_cleaned.withColumn(
    "Stock",
    when(
        regexp_extract(col("RawStock"), r"(\d+)", 1) != "",
        regexp_extract(col("RawStock"), r"(\d+)", 1).cast("int")
    )
    .when(lower(trim(col("RawStock"))) == "twenty five", 25)
    .when(lower(trim(col("RawStock"))) == "twenty", 20)
    .when(lower(trim(col("RawStock"))) == "eighteen", 18)
    .when(lower(trim(col("RawStock"))) == "fifteen", 15)
    .when(lower(trim(col("RawStock"))) == "twelve", 12)
    .otherwise(None)
    .cast("int")
)

# Normalize dates
df_inventory_cleaned = df_inventory_cleaned.withColumn(
    "LastStocked",
    coalesce(
        to_date(col("LastStocked"), "yyyy/MM/dd"),
        to_date(col("LastStocked"), "dd-MM-yyyy"),
        to_date(col("LastStocked"), "yyyy.MM.dd"),
        to_date(col("LastStocked"), "dd/MM/yyyy"),
        to_date(col("LastStocked"), "MM-dd-yyyy"),
        to_date(col("LastStocked"), "yyyy-MM-dd")
    )
)

# Cost price
df_inventory_cleaned = df_inventory_cleaned.withColumn(
    "CostPrice",
    regexp_extract(col("CostPrice"), r"(\d+(?:\.\d+)?)", 1)
    .cast("double")
)

# Warehouse
df_inventory_cleaned = df_inventory_cleaned.withColumn(
    "Warehouse",
    initcap(
        trim(
            regexp_replace(col("RawWarehouse"), r"[^a-zA-Z0-9\s]", " ")
        )
    )
)

# Available
df_inventory_cleaned = df_inventory_cleaned.withColumn(
    "Available",
    when(lower(trim(col("RawAvailable"))).isin("yes", "y", "true"), True)
    .when(lower(trim(col("RawAvailable"))).isin("no", "n", "false"), False)
    .otherwise(None)
)

# Keep only final Silver columns
df_inventory_cleaned = df_inventory_cleaned.select(
    "ProductID",
    "ProductName",
    "Stock",
    "LastStocked",
    "Warehouse",
    "CostPrice",
    "Available"
)

# Remove duplicate products
df_inventory_cleaned = df_inventory_cleaned.dropDuplicates(["ProductID"])

# Save Silver table
df_inventory_cleaned.write \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .format("delta") \
    .saveAsTable("silver_inventory")

# Check result
display(df_inventory_cleaned)

StatementMeta(, 2e424b1e-5c63-404a-b421-0f94eac6cf48, 55, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 37c3c410-f361-4bd7-9210-bbc1d3c807db)

In [77]:
display(df_inventory_cleaned)

StatementMeta(, 2e424b1e-5c63-404a-b421-0f94eac6cf48, 56, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 5aaf8c33-3b99-4d27-bd2c-dc29fa9b3f01)

In [78]:
spark.table("silver_inventory").printSchema()


StatementMeta(, 2e424b1e-5c63-404a-b421-0f94eac6cf48, 57, Finished, Available, Finished, False)

root
 |-- ProductID: string (nullable = true)
 |-- ProductName: string (nullable = true)
 |-- Stock: integer (nullable = true)
 |-- LastStocked: date (nullable = true)
 |-- Warehouse: string (nullable = true)
 |-- CostPrice: double (nullable = true)
 |-- Available: boolean (nullable = true)



In [79]:
from pyspark.sql.functions import *
from pyspark.sql.types import DoubleType

# ============================================================
# 1. Rename Prop_* columns using the actual Returns structure
# ============================================================

df_returns_cleaned = (
    df_returns
    .withColumnRenamed("Prop_0", "ReturnID")
    .withColumnRenamed("Prop_1", "OrderID")
    .withColumnRenamed("Prop_2", "CustomerID")
    .withColumnRenamed("Prop_3", "Product")
    .withColumnRenamed("Prop_4", "ReturnReason")
    .withColumnRenamed("Prop_5", "ReturnDate")
    .withColumnRenamed("Prop_6", "RefundStatus")
    .withColumnRenamed("Prop_7", "PickupAddress")
    .withColumnRenamed("Prop_8", "ReturnAmount")
)

# ============================================================
# 2. Remove the accidental header row
# ============================================================

df_returns_cleaned = df_returns_cleaned.filter(
    lower(trim(col("ReturnID"))) != "return_id"
)

# ============================================================
# 3. Clean ReturnID
# ============================================================

df_returns_cleaned = df_returns_cleaned.withColumn(
    "ReturnID",
    trim(upper(col("ReturnID")))
)

# ============================================================
# 4. Clean OrderID
# ============================================================

df_returns_cleaned = df_returns_cleaned.withColumn(
    "OrderID",
    trim(upper(col("OrderID")))
)

# ============================================================
# 5. Clean CustomerID
# ============================================================

df_returns_cleaned = df_returns_cleaned.withColumn(
    "CustomerID",
    trim(upper(col("CustomerID")))
)

# ============================================================
# 6. Clean Product
# ============================================================

df_returns_cleaned = df_returns_cleaned.withColumn(
    "Product",
    initcap(
        trim(
            regexp_replace(
                col("Product"),
                r"[^a-zA-Z0-9\s]",
                " "
            )
        )
    )
)

# ============================================================
# 7. Clean ReturnReason
# ============================================================

df_returns_cleaned = df_returns_cleaned.withColumn(
    "ReturnReason",
    initcap(
        trim(
            regexp_replace(
                col("ReturnReason"),
                r"[^a-zA-Z0-9\s]",
                " "
            )
        )
    )
)

# ============================================================
# 8. Clean ReturnDate
# ============================================================

df_returns_cleaned = df_returns_cleaned.withColumn(
    "ReturnDate",
    coalesce(
        to_date(col("ReturnDate"), "yyyy/MM/dd"),
        to_date(col("ReturnDate"), "dd-MM-yyyy"),
        to_date(col("ReturnDate"), "yyyy.MM.dd"),
        to_date(col("ReturnDate"), "dd/MM/yyyy"),
        to_date(col("ReturnDate"), "MM-dd-yyyy"),
        to_date(col("ReturnDate"), "yyyy-MM-dd")
    )
)

# ============================================================
# 9. Clean RefundStatus
# ============================================================

df_returns_cleaned = df_returns_cleaned.withColumn(
    "RefundStatus",
    lower(
        trim(
            regexp_replace(
                col("RefundStatus"),
                r"[^a-zA-Z\s]",
                ""
            )
        )
    )
)

# ============================================================
# 10. Clean PickupAddress
# ============================================================

df_returns_cleaned = df_returns_cleaned.withColumn(
    "PickupAddress",
    initcap(
        trim(
            regexp_replace(
                col("PickupAddress"),
                r"[^a-zA-Z0-9\s]",
                " "
            )
        )
    )
)

# ============================================================
# 11. Clean ReturnAmount
# ============================================================

df_returns_cleaned = df_returns_cleaned.withColumn(
    "ReturnAmount",
    regexp_extract(
        col("ReturnAmount"),
        r"(\d+(?:\.\d+)?)",
        1
    ).cast(DoubleType())
)

# ============================================================
# 12. Remove invalid records
# ============================================================

df_returns_cleaned = df_returns_cleaned.filter(
    col("ReturnID").isNotNull()
)

# ============================================================
# 13. Remove duplicate Return IDs
# ============================================================

df_returns_cleaned = df_returns_cleaned.dropDuplicates(
    ["ReturnID"]
)

# ============================================================
# 14. Display cleaned data
# ============================================================

display(df_returns_cleaned)

StatementMeta(, 2e424b1e-5c63-404a-b421-0f94eac6cf48, 58, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, aa125c92-d73d-4372-ac61-52906ce9fded)

In [81]:
display(df_returns_cleaned)

StatementMeta(, 2e424b1e-5c63-404a-b421-0f94eac6cf48, 60, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 54f43d1b-4645-4075-8fdd-c6f1ba2f6f36)

In [84]:
from pyspark.sql.functions import *

# ============================================================
# 1. Load Silver tables
# ============================================================

df_orders = spark.table("silver_orders")
df_returns = spark.table("silver_returns")
df_inventory = spark.table("silver_inventory")


# ============================================================
# 2. Standardize Returns column names
# ============================================================

df_returns = (
    df_returns
    .withColumnRenamed("Return_ID", "ReturnID")
    .withColumnRenamed("Order_ID", "OrderID")
    .withColumnRenamed("Customer_ID", "CustomerID")
    .withColumnRenamed("Return_Reason", "ReturnReason")
    .withColumnRenamed("Return_Date", "ReturnDate")
    .withColumnRenamed("Refund_Status", "RefundStatus")
    .withColumnRenamed("Pickup_Address", "PickupAddress")
    .withColumnRenamed("Return_Amount", "ReturnAmount")
)


# ============================================================
# 3. Aggregate Returns by OrderID
# ============================================================

df_return_summary = (
    df_returns
    .groupBy("OrderID")
    .agg(
        countDistinct("ReturnID").alias("Return_Count"),
        first("ReturnDate", ignorenulls=True).alias("ReturnDate"),
        first("RefundStatus", ignorenulls=True).alias("RefundStatus"),
        round(sum("ReturnAmount"), 2).alias("Total_Return_Amount")
    )
)


# ============================================================
# 4. JOIN Orders + Returns
# Orders uses Order_ID
# Returns uses OrderID
# ============================================================

df_order_return = (
    df_orders.alias("o")
    .join(
        df_return_summary.alias("r"),
        col("o.Order_ID") == col("r.OrderID"),
        "left"
    )
    .select(
        col("o.*"),
        col("r.Return_Count"),
        col("r.ReturnDate"),
        col("r.RefundStatus"),
        col("r.Total_Return_Amount")
    )
)


# ============================================================
# 5. Replace NULL return values
# ============================================================

df_order_return = (
    df_order_return
    .withColumn(
        "Return_Count",
        coalesce(col("Return_Count"), lit(0))
    )
    .withColumn(
        "Total_Return_Amount",
        coalesce(col("Total_Return_Amount"), lit(0.0))
    )
)


# ============================================================
# 6. Join Inventory using ProductName
# ============================================================

df_enriched = (
    df_order_return.alias("o")
    .join(
        df_inventory.alias("i"),
        col("o.ProductName") == col("i.ProductName"),
        "left"
    )
    .select(
        col("o.*"),
        col("i.Stock"),
        col("i.CostPrice"),
        col("i.Warehouse"),
        col("i.Available")
    )
)


# ============================================================
# 7. Calculate COGS
# COGS = Quantity Sold × Cost Price
# ============================================================

df_enriched = df_enriched.withColumn(
    "COGS",
    col("Qty") * col("CostPrice")
)


# ============================================================
# 8. Calculate Net Profit
# ============================================================

df_enriched = df_enriched.withColumn(
    "Net_Profit",
    col("OrderAmount") - col("COGS")
)


# ============================================================
# 9. Product-Level KPIs
# ============================================================

df_kpi = (
    df_enriched
    .groupBy("ProductName")
    .agg(

        countDistinct("Order_ID")
            .alias("Total_Orders"),

        countDistinct("CustomerID")
            .alias("Unique_Customers"),

        sum("Qty")
            .alias("Total_Quantity_Sold"),

        round(sum("OrderAmount"), 2)
            .alias("Total_Revenue"),

        round(avg("OrderAmount"), 2)
            .alias("Avg_Order_Value"),

        round(sum("COGS"), 2)
            .alias("Total_COGS"),

        round(avg("CostPrice"), 2)
            .alias("Avg_Cost"),

        round(sum("Net_Profit"), 2)
            .alias("Net_Profit"),

        sum("Return_Count")
            .alias("Total_Returns"),

        countDistinct(
            when(
                col("Return_Count") > 0,
                col("Order_ID")
            )
        ).alias("Returned_Orders"),

        round(sum("Total_Return_Amount"), 2)
            .alias("Total_Return_Amount"),

        max("Stock")
            .alias("Current_Stock")
    )
)


# ============================================================
# 10. Calculate Return Rate
# ============================================================

df_kpi = df_kpi.withColumn(
    "Return_Rate_Percent",
    round(
        (col("Returned_Orders") / col("Total_Orders")) * 100,
        2
    )
)


# ============================================================
# 11. Save Gold table
# ============================================================

df_kpi.write \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .format("delta") \
    .saveAsTable("gold_product_kpis")


# ============================================================
# 12. Display Gold table
# ============================================================

display(df_kpi)

StatementMeta(, 2e424b1e-5c63-404a-b421-0f94eac6cf48, 63, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, c0123ba8-76f6-49a6-bc38-f5e94e6c71ad)

In [85]:
display(df_kpi)

StatementMeta(, 2e424b1e-5c63-404a-b421-0f94eac6cf48, 64, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 962e210a-c7cb-41d1-b593-9ea2e539f484)

In [86]:
from notebookutils import mssparkutils

# Bronze
mssparkutils.fs.mkdirs("Files/Bronze/Orders")
mssparkutils.fs.mkdirs("Files/Bronze/Returns")
mssparkutils.fs.mkdirs("Files/Bronze/Inventory")

# Silver
mssparkutils.fs.mkdirs("Files/Silver/Orders")
mssparkutils.fs.mkdirs("Files/Silver/Returns")
mssparkutils.fs.mkdirs("Files/Silver/Inventory")

# Gold
mssparkutils.fs.mkdirs("Files/Gold/Product_KPIs")

print("Folder structure created successfully!")

StatementMeta(, 2e424b1e-5c63-404a-b421-0f94eac6cf48, 65, Finished, Available, Finished, False)

Folder structure created successfully!


In [ ]:
df_order.write.mode("overwrite").format("delta").save("Files/Bronze/Orders")

df_returns.write.mode("overwrite").format("delta").save("Files/Bronze/Returns")

df_inventory.write.mode("overwrite").format("delta").save("Files/Bronze/Inventory")

StatementMeta(, 2e424b1e-5c63-404a-b421-0f94eac6cf48, 66, Finished, Available, Finished, False)

In [ ]:
df_orders.write.mode("overwrite").format("delta").save("Files/Silver/Orders")

df_returns_cleaned.write.mode("overwrite").format("delta").save("Files/Silver/Returns")

df_inventory_cleaned.write.mode("overwrite").format("delta").save("Files/Silver/Inventory")

StatementMeta(, 2e424b1e-5c63-404a-b421-0f94eac6cf48, 67, Finished, Available, Finished, False)

In [ ]:
df_kpi.write \
    .mode("overwrite") \
    .format("delta") \
    .save("Files/Gold/Product_KPIs")

StatementMeta(, 2e424b1e-5c63-404a-b421-0f94eac6cf48, 68, Finished, Available, Finished, False)

In [ ]:
from pyspark.sql.functions import *

# Load Silver tables
df_orders = spark.table("silver_orders")
df_returns = spark.table("silver_returns")
df_inventory = spark.table("silver_inventory")

# Rename Returns columns to match our Gold logic
df_returns = (
    df_returns
    .withColumnRenamed("Return_ID", "ReturnID")
    .withColumnRenamed("Order_ID", "OrderID")
    .withColumnRenamed("Customer_ID", "CustomerID")
    .withColumnRenamed("Return_Reason", "ReturnReason")
    .withColumnRenamed("Return_Date", "ReturnDate")
    .withColumnRenamed("Refund_Status", "RefundStatus")
    .withColumnRenamed("Pickup_Address", "PickupAddress")
    .withColumnRenamed("Return_Amount", "ReturnAmount")
)

# Summarize returns per order
df_return_summary = (
    df_returns
    .groupBy("OrderID")
    .agg(
        countDistinct("ReturnID").alias("Return_Count"),
        first("ReturnDate", ignorenulls=True).alias("ReturnDate"),
        first("RefundStatus", ignorenulls=True).alias("RefundStatus"),
        round(sum("ReturnAmount"), 2).alias("Total_Return_Amount")
    )
)

# Join Orders + Returns
df_order_return = (
    df_orders.alias("o")
    .join(
        df_return_summary.alias("r"),
        col("o.Order_ID") == col("r.OrderID"),
        "left"
    )
    .select(
        col("o.*"),
        col("r.Return_Count"),
        col("r.ReturnDate"),
        col("r.RefundStatus"),
        col("r.Total_Return_Amount")
    )
    .withColumn(
        "Return_Count",
        coalesce(col("Return_Count"), lit(0))
    )
    .withColumn(
        "Total_Return_Amount",
        coalesce(col("Total_Return_Amount"), lit(0.0))
    )
)

# Join Inventory
df_enriched = (
    df_order_return.alias("o")
    .join(
        df_inventory.alias("i"),
        col("o.ProductName") == col("i.ProductName"),
        "left"
    )
    .select(
        col("o.*"),
        col("i.Stock"),
        col("i.CostPrice"),
        col("i.Warehouse"),
        col("i.Available")
    )
)

# Calculate COGS
df_enriched = df_enriched.withColumn(
    "COGS",
    col("Qty") * col("CostPrice")
)

# Calculate Net Profit
df_enriched = df_enriched.withColumn(
    "Net_Profit",
    col("OrderAmount") - col("COGS")
)

print("df_enriched created successfully")

StatementMeta(, 11b64e5d-965f-45e7-8655-4485c8f5b240, 4, Finished, Available, Finished, False)

df_enriched created successfully


In [ ]:
display(
    df_enriched.select(
        "Order_ID",
        "ProductName",
        "Qty",
        "OrderAmount",
        "CostPrice",
        "COGS",
        "Net_Profit"
    ).orderBy("Order_ID")
)

StatementMeta(, 11b64e5d-965f-45e7-8655-4485c8f5b240, 5, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 4c4f5509-0147-4b5a-ae49-e15b2caecbba)

In [ ]:
display(
    df_inventory.select(
        "ProductID",
        "ProductName",
        "CostPrice"
    ).orderBy("ProductName")
)

StatementMeta(, 11b64e5d-965f-45e7-8655-4485c8f5b240, 6, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, b2b3a804-ee41-4d91-a4f1-93f6d9064331)

In [ ]:
from pyspark.sql.functions import *

df_enriched = df_enriched.withColumn(
    "Profit_Status",
    when(
        col("CostPrice").isNull(),
        "Cost Data Missing"
    )
    .when(
        col("Qty").isNull(),
        "Quantity Missing"
    )
    .when(
        col("Net_Profit") < 0,
        "Loss"
    )
    .otherwise("Profit")
)

display(
    df_enriched.select(
        "Order_ID",
        "ProductName",
        "Qty",
        "OrderAmount",
        "CostPrice",
        "COGS",
        "Net_Profit",
        "Profit_Status"
    )
)

StatementMeta(, 11b64e5d-965f-45e7-8655-4485c8f5b240, 7, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, d7d01286-3743-4a24-8b44-cfa13f652745)

In [ ]:
from pyspark.sql.functions import *

# Add profit status
df_enriched = df_enriched.withColumn(
    "Profit_Status",
    when(col("CostPrice").isNull(), "Cost Data Missing")
    .when(col("Qty").isNull(), "Quantity Missing")
    .when(col("Net_Profit") < 0, "Loss")
    .otherwise("Profit")
)

display(
    df_enriched.select(
        "Order_ID",
        "ProductName",
        "Qty",
        "OrderAmount",
        "CostPrice",
        "COGS",
        "Net_Profit",
        "Profit_Status"
    )
)

StatementMeta(, 11b64e5d-965f-45e7-8655-4485c8f5b240, 8, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, a78c8e0d-d796-4d1a-8353-5752c3c7e400)

In [ ]:
df_kpi = (
    df_enriched
    .groupBy("ProductName")
    .agg(
        countDistinct("Order_ID").alias("Total_Orders"),
        countDistinct("CustomerID").alias("Unique_Customers"),
        sum("Qty").alias("Total_Quantity_Sold"),
        round(sum("OrderAmount"), 2).alias("Total_Revenue"),
        round(avg("OrderAmount"), 2).alias("Avg_Order_Value"),
        round(sum("COGS"), 2).alias("Total_COGS"),
        round(avg("CostPrice"), 2).alias("Avg_Cost"),
        round(sum("Net_Profit"), 2).alias("Net_Profit"),
        sum("Return_Count").alias("Total_Returns"),
        countDistinct(
            when(col("Return_Count") > 0, col("Order_ID"))
        ).alias("Returned_Orders"),
        round(sum("Total_Return_Amount"), 2).alias("Total_Return_Amount"),
        max("Stock").alias("Current_Stock"),
        first("Profit_Status", ignorenulls=True).alias("Profit_Status")
    )
)

df_kpi = df_kpi.withColumn(
    "Return_Rate_Percent",
    round(
        (col("Returned_Orders") / col("Total_Orders")) * 100,
        2
    )
)

StatementMeta(, 11b64e5d-965f-45e7-8655-4485c8f5b240, 9, Finished, Available, Finished, False)

In [ ]:
df_kpi.write \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .format("delta") \
    .saveAsTable("gold_product_kpis")

StatementMeta(, 11b64e5d-965f-45e7-8655-4485c8f5b240, 10, Finished, Available, Finished, False)

In [ ]:
df_kpi.write \
    .mode("overwrite") \
    .format("delta") \
    .save("Files/Gold/Product_KPIs")

StatementMeta(, 11b64e5d-965f-45e7-8655-4485c8f5b240, 11, Finished, Available, Finished, False)

AnalysisException: [_LEGACY_ERROR_TEMP_DELTA_0007] A schema mismatch detected when writing to the Delta table (Table ID: 92976806-bf53-480d-9d96-98a602c318ac).
To enable schema migration using DataFrameWriter or DataStreamWriter, please set:
'.option("mergeSchema", "true")'.
For other operations, set the session configuration
spark.databricks.delta.schema.autoMerge.enabled to "true". See the documentation
specific to the operation for details.

Table schema:
root
-- ProductName: string (nullable = true)
-- Total_Orders: long (nullable = true)
-- Unique_Customers: long (nullable = true)
-- Total_Quantity_Sold: long (nullable = true)
-- Total_Revenue: double (nullable = true)
-- Avg_Order_Value: double (nullable = true)
-- Total_COGS: double (nullable = true)
-- Avg_Cost: double (nullable = true)
-- Net_Profit: double (nullable = true)
-- Total_Returns: long (nullable = true)
-- Returned_Orders: long (nullable = true)
-- Total_Return_Amount: double (nullable = true)
-- Current_Stock: int

In [ ]:
display(spark.table("gold_product_kpis"))

StatementMeta(, 11b64e5d-965f-45e7-8655-4485c8f5b240, 12, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 11d7f753-3a82-46c6-b3b0-95ef8eb4a37a)